# E21: Atlas PerturbOT R2 and MMD

**Started:** June 22, 2026

**Last updated:** June 22, 2026

**Research questions:** Do MLPs learned via Perturb-OT have better R2 and MMD on held-out monocytes and CD8s than 1. random within-cell type pairs, 2. EGW pairing, and 3. cross-species scGen?

**Hypothesis:** Perturb-OT will outperform random pairs (fourth place), EGW pairing (second place), and cross-species scGen (third place) because it uses both the least-effort principle and cell type restriction during optimization.

**Conclusion:** TBD.

**Potential Next Steps:** TBD.

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
#import scvi
from scipy import sparse
import sys
import os
sys.path.append(os.path.abspath(".."))
from speciesot_helpers import match_cells_by_celltype_tissue, mouse_human_orthologs_biomart
import perturbot
from perturbot.match import get_coupling_egw_labels_ott
from perturbot.predict import train_mlp

/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/lightning/fabric/__init__.py:40: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/l

ModuleNotFoundError: No module named 'jaxlib.xla_extension'

In [2]:
MOUSE_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_muris/sampled_mouse_shared.h5ad"
HUMAN_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_sapiens/sampled_human_shared.h5ad"

In [3]:
mouse_full = sc.read_h5ad(MOUSE_H5AD)
human_full = sc.read_h5ad(HUMAN_H5AD)

mouse_B = mouse_full.raw.to_adata()
human_B = human_full.raw.to_adata()

mouse_B.obs = mouse_full.obs.copy()
human_B.obs = human_full.obs.copy()

mouse_B = mouse_B[mouse_B.obs['assay']=="10x 3' v2"]
human_B = human_B[human_B.obs['assay']=="10x 3' v3"]

In [4]:
human_B.obs["cell_type"] = human_B.obs["cell_type"].cat.add_categories(["HSPC"])
human_B.obs.loc[human_B.obs["cell_type"].isin(["hematopoietic precursor cell","hematopoietic stem cell"]), "cell_type"] = "HSPC"

/tmp/ipykernel_2822540/3064969281.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  human_B.obs["cell_type"] = human_B.obs["cell_type"].cat.add_categories(["HSPC"])


In [5]:
mouse_matched_B, human_matched_B = match_cells_by_celltype_tissue(
    mouse_B, human_B,
    cell_type_key="cell_type_ontology_term_id",
    tissue_key="tissue_ontology_term_id",
)

for a in [mouse_matched_B, human_matched_B]:
    a.layers["counts"] = a.X.copy()
    a.X = a.X.astype("float32")
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)

In [7]:
mouse_matched_B

AnnData object with n_obs × n_vars = 4027 × 18024
    obs: 'batch', 'tissue_FACS_droplet', 'free_annotation', 'n_counts', 'n_genes', 'louvain', 'leiden', 'age', 'method', 'donor_id', 'subtissue', 'tissue_free_annotation', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'suspension_type', 'FACS.selection', 'tissue_original', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'shared_cell_type'
    var: 'index', 'n_cells-0', 'n_cells-1', 'gene_symbols', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'age_colors', 'citation', 'leiden', 'louvain', 'louvain_colors', 'method_colors', 'neighbors', 'organism', 'organism_ontology_term_id', 'pca', 'schema_reference', 'schema_ver